In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Transform dữ liệu
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

# Dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [16]:
class CNN_Model(nn.Module):
    def __init__(self):
        super(CNN_Model, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)

        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128*4*4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self,x):

        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))

        x = x.view(-1,128*4*4)

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_Model().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
num_epochs = 50

for epoch in range(num_epochs):

    running_loss = 0

    for images,labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} Loss:{running_loss/len(train_loader):.4f}")

Epoch 1/50 Loss:0.4015
Epoch 2/50 Loss:0.3909
Epoch 3/50 Loss:0.3758
Epoch 4/50 Loss:0.3746
Epoch 5/50 Loss:0.3589
Epoch 6/50 Loss:0.3523
Epoch 7/50 Loss:0.3465
Epoch 8/50 Loss:0.3368
Epoch 9/50 Loss:0.3235
Epoch 10/50 Loss:0.3228
Epoch 11/50 Loss:0.3136
Epoch 12/50 Loss:0.3040
Epoch 13/50 Loss:0.3007
Epoch 14/50 Loss:0.2954
Epoch 15/50 Loss:0.2907
Epoch 16/50 Loss:0.2882
Epoch 17/50 Loss:0.2797
Epoch 18/50 Loss:0.2741
Epoch 19/50 Loss:0.2655
Epoch 20/50 Loss:0.2652
Epoch 21/50 Loss:0.2549
Epoch 22/50 Loss:0.2565
Epoch 23/50 Loss:0.2469
Epoch 24/50 Loss:0.2480
Epoch 25/50 Loss:0.2509
Epoch 26/50 Loss:0.2426
Epoch 27/50 Loss:0.2281
Epoch 28/50 Loss:0.2343
Epoch 29/50 Loss:0.2326
Epoch 30/50 Loss:0.2304
Epoch 31/50 Loss:0.2262
Epoch 32/50 Loss:0.2201
Epoch 33/50 Loss:0.2234
Epoch 34/50 Loss:0.2170
Epoch 35/50 Loss:0.2133
Epoch 36/50 Loss:0.2114
Epoch 37/50 Loss:0.2130
Epoch 38/50 Loss:0.2075
Epoch 39/50 Loss:0.2123
Epoch 40/50 Loss:0.2068
Epoch 41/50 Loss:0.2007
Epoch 42/50 Loss:0.1995
E

In [26]:
correct = 0
total = 0

with torch.no_grad():

    for images,labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _,predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Accuracy:",100*correct/total,"%")

Accuracy: 77.46 %
